# NB05 — Final Clustering dengan Parameter Terbaik

**Tujuan:** Menjalankan clustering final menggunakan parameter terbaik dari NB04 dan mengevaluasi hasilnya secara komprehensif.

**Parameter terbaik (NB04 v3):**
| Parameter | Nilai |
|-----------|-------|
| `min_cluster_size` | 50 |
| `min_samples` | 5 |
| `cluster_selection_method` | `'eom'` |
| `metric` | `'euclidean'` (pada L2-normalized embeddings ≡ cosine) |
| `approx_min_span_tree` | `True` |

**Hasil yang diharapkan:** n_clusters ≈ 144, coverage ≈ 92.4%

**Input:** `output_nb01/embeddings.npy`, `output_nb01/metadata.pkl`  
**Output:** `output_nb05/labels.npy`, `output_nb05/metadata_labeled.pkl`, `output_nb05/cluster_summary.pkl`

## 0. Instalasi & Import

In [ ]:
import subprocess, sys

subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "hdbscan", "scikit-learn", "matplotlib", "pandas", "umap-learn", "-q"],
    check=True
)
print("✅ Dependensi siap")

In [ ]:
import pickle
import time
import warnings
from collections import Counter
from pathlib import Path

import hdbscan
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
import umap
from importlib.metadata import version
from sklearn.metrics import silhouette_score, davies_bouldin_score

warnings.filterwarnings("ignore")

print(f"hdbscan  : {version('hdbscan')}")
print(f"umap     : {version('umap-learn')}")
print(f"sklearn  : {version('scikit-learn')}")

## 1. Konfigurasi

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

In [ ]:
# ─── PATH ─────────────────────────────────────────────────────────────────────
INPUT_DIR  = Path("/content/drive/MyDrive/OTW S.KOM/Embeddings/output_nb01")
OUTPUT_DIR = Path("/content/drive/MyDrive/OTW S.KOM/Embeddings/output_nb05")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ─── Parameter Terbaik dari NB04 v3 ───────────────────────────────────────────
MIN_CLUSTER_SIZE         = 50
MIN_SAMPLES              = 5
CLUSTER_SELECTION_METHOD = "eom"
# metric = 'euclidean' pada L2-normalized embedding ≡ cosine distance

# ─── Evaluasi ─────────────────────────────────────────────────────────────────
RANDOM_SEED = 42
SUBSAMPLE   = 2000   # subsample untuk Silhouette & DBI (O(N²) terlalu besar)

# ─── Target ───────────────────────────────────────────────────────────────────
TARGET_MAX_CLUSTERS = 150
MIN_COVERAGE        = 70.0

print("=== Konfigurasi NB05 ===")
print(f"  Input          : {INPUT_DIR}")
print(f"  Output         : {OUTPUT_DIR}")
print()
print(f"  min_cluster_size         = {MIN_CLUSTER_SIZE}")
print(f"  min_samples              = {MIN_SAMPLES}")
print(f"  cluster_selection_method = '{CLUSTER_SELECTION_METHOD}'")
print(f"  metric                   = 'euclidean'  (L2-normalized embeddings)")
print()
print(f"  Target: n_clusters < {TARGET_MAX_CLUSTERS},  coverage >= {MIN_COVERAGE}%")

## 2. Load & Normalize Embeddings

In [ ]:
# ── Load embeddings ────────────────────────────────────────────────────────────
embeddings = np.load(INPUT_DIR / "embeddings.npy").astype(np.float32)
N, D = embeddings.shape

with open(INPUT_DIR / "metadata.pkl", "rb") as f:
    metadata = pickle.load(f)

print(f"Embeddings : {embeddings.shape}  dtype={embeddings.dtype}")
print(f"Metadata   : {len(metadata)} records")

# ── L2-normalize ───────────────────────────────────────────────────────────────
# euclidean distance pada L2-normalized embedding ≡ cosine distance
norms    = np.linalg.norm(embeddings, axis=1, keepdims=True)
emb_norm = embeddings / norms

norm_check = np.linalg.norm(emb_norm, axis=1)
print(f"\nL2-norm check — min={norm_check.min():.6f}, max={norm_check.max():.6f}  "
      f"(harus ≈ 1.0)")

## 3. Clustering Final

Direct HDBSCAN pada embedding ternormalisasi.  
Gunakan `core_dist_n_jobs=-1` (bukan `n_jobs`) agar tidak konflik dengan KDTree.

In [ ]:
print("Menjalankan HDBSCAN final...")
print(f"  mcs={MIN_CLUSTER_SIZE}, ms={MIN_SAMPLES}, csm='{CLUSTER_SELECTION_METHOD}', "
      f"metric='euclidean'")
print(f"  N={N:,} embeddings × D={D} dimensi")
print()

t0 = time.time()

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=MIN_CLUSTER_SIZE,
    min_samples=MIN_SAMPLES,
    metric="euclidean",
    cluster_selection_method=CLUSTER_SELECTION_METHOD,
    approx_min_span_tree=True,
    core_dist_n_jobs=-1,
)
clusterer.fit(emb_norm)

t_cluster = time.time() - t0

labels     = clusterer.labels_
n_clusters = int(len(set(labels[labels >= 0])))
n_noise    = int((labels == -1).sum())
n_clustered = int((labels >= 0).sum())
coverage   = n_clustered / N * 100
noise_pct  = n_noise / N * 100

target_ok = "✅" if n_clusters < TARGET_MAX_CLUSTERS and coverage >= MIN_COVERAGE else "⚠️"

print(f"Selesai dalam {t_cluster:.0f}s ({t_cluster/60:.1f} menit)")
print()
print(f"  n_clusters  : {n_clusters}  {target_ok}")
print(f"  coverage    : {coverage:.2f}%  {target_ok}")
print(f"  noise       : {noise_pct:.2f}%  ({n_noise:,} foto)")
print(f"  n_clustered : {n_clustered:,} foto")

## 4. Evaluasi Kuantitatif

### 4a. Statistik Distribusi Cluster

In [ ]:
cluster_sizes = Counter(labels[labels >= 0])
sizes = np.array(sorted(cluster_sizes.values(), reverse=True))

print("=== Distribusi Ukuran Cluster ===")
print(f"  Jumlah cluster       : {n_clusters}")
print(f"  Ukuran terbesar      : {sizes.max()} foto")
print(f"  Ukuran terkecil      : {sizes.min()} foto")
print(f"  Rata-rata            : {sizes.mean():.1f} foto/cluster")
print(f"  Median               : {np.median(sizes):.1f} foto/cluster")
print(f"  Std dev              : {sizes.std():.1f}")
print()
print("  Distribusi bucket ukuran:")
for lo, hi in [(1,9),(10,24),(25,49),(50,99),(100,199),(200,999)]:
    cnt = int(((sizes >= lo) & (sizes <= hi)).sum())
    if cnt > 0:
        print(f"    [{lo:3}–{hi:3}] foto : {cnt} cluster")
print()

# Top-10 cluster terbesar
top10 = sorted(cluster_sizes.items(), key=lambda x: x[1], reverse=True)[:10]
print("  Top-10 cluster terbesar:")
for rank, (cid, sz) in enumerate(top10, 1):
    print(f"    #{rank:2}  cluster {cid:4}  →  {sz:4} foto")

### 4b. Silhouette Score (Subsample)

Mengukur separasi antar cluster. Range **[-1, 1]**, makin tinggi makin baik.  
Subsample digunakan karena Silhouette Score bersifat O(N²).

In [ ]:
t0 = time.time()

rng      = np.random.default_rng(seed=RANDOM_SEED)
idx_pool = np.where(labels >= 0)[0]
n_sub    = min(SUBSAMPLE, len(idx_pool))
idx_sub  = rng.choice(idx_pool, size=n_sub, replace=False)
idx_sub  = np.sort(idx_sub)

emb_sub    = emb_norm[idx_sub].astype(np.float64)
labels_sub = labels[idx_sub]

# Filter cluster dengan < 2 titik di subsample
label_counts  = Counter(labels_sub)
valid_labels  = {lbl for lbl, cnt in label_counts.items() if cnt >= 2}
mask_valid    = np.isin(labels_sub, list(valid_labels))
emb_sub_v     = emb_sub[mask_valid]
labels_sub_v  = labels_sub[mask_valid]

sil_score = silhouette_score(emb_sub_v, labels_sub_v, metric="euclidean")
t_sil = time.time() - t0

print(f"Silhouette Score : {sil_score:.4f}  ({t_sil:.1f} detik)")
print(f"  Subsample      : {n_sub:,} dari {n_clustered:,} titik (seed={RANDOM_SEED})")
print(f"  Cluster valid  : {len(valid_labels)} dari {n_clusters}")
print()
if sil_score >= 0.5:
    print("  Interpretasi: ✅ Cluster terpisah dengan baik (≥ 0.5)")
elif sil_score >= 0.25:
    print("  Interpretasi: ⚠️ Cluster cukup terpisah (0.25–0.5)")
else:
    print("  Interpretasi: ❌ Cluster overlap atau struktur lemah (< 0.25)")

### 4c. Davies-Bouldin Index (Subsample)

Mengukur rasio dispersi dalam cluster vs antar cluster.  
Range **[0, ∞)** — makin **rendah** makin baik. Nilai < 1.0 dianggap baik.

In [ ]:
t0 = time.time()

dbi_score = davies_bouldin_score(emb_sub_v, labels_sub_v)
t_dbi = time.time() - t0

print(f"Davies-Bouldin Index : {dbi_score:.4f}  ({t_dbi:.1f} detik)")
print(f"  Range [0, ∞) — makin RENDAH makin baik")
print()
if dbi_score < 1.0:
    print("  Interpretasi: ✅ Cluster berkualitas baik (< 1.0)")
elif dbi_score < 2.0:
    print("  Interpretasi: ⚠️ Cluster cukup baik (1.0–2.0)")
else:
    print("  Interpretasi: ❌ Cluster overlap atau kualitas rendah (≥ 2.0)")

### 4d. Ringkasan Semua Metrik

In [ ]:
print("=" * 58)
print("  RINGKASAN METRIK EVALUASI — NB05 FINAL CLUSTERING")
print("=" * 58)
print(f"  Dataset            : {N:,} foto × {D} dimensi")
print(f"  Subsample evaluasi : {n_sub:,} titik (seed={RANDOM_SEED})")
print()
print(f"  Parameter clustering:")
print(f"    min_cluster_size         = {MIN_CLUSTER_SIZE}")
print(f"    min_samples              = {MIN_SAMPLES}")
print(f"    cluster_selection_method = '{CLUSTER_SELECTION_METHOD}'")
print(f"    metric                   = 'euclidean' (L2-norm ≡ cosine)")
print()
print(f"  Hasil clustering:")
print(f"    n_clusters    : {n_clusters}  (target < {TARGET_MAX_CLUSTERS})"
      f"  {'✅' if n_clusters < TARGET_MAX_CLUSTERS else '⚠️'}")
print(f"    Coverage      : {coverage:.2f}%  (target ≥ {MIN_COVERAGE}%)"
      f"  {'✅' if coverage >= MIN_COVERAGE else '⚠️'}")
print(f"    Noise rate    : {noise_pct:.2f}%  ({n_noise:,} foto)")
print(f"    Ukuran rata²  : {sizes.mean():.1f} foto/cluster")
print(f"    Ukuran median : {np.median(sizes):.1f} foto/cluster")
print()
print(f"  Metrik kualitas clustering:")
print(f"    Silhouette Score      : {sil_score:.4f}  (↑ lebih baik, range [-1,1])")
print(f"    Davies-Bouldin Index  : {dbi_score:.4f}  (↓ lebih baik, range [0,∞))")
print(f"    DBCV                  : N/A  (approx_min_span_tree=True)")
print("=" * 58)

## 5. Visualisasi Distribusi Cluster

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(
    f"NB05 — Distribusi Cluster  "
    f"(n={n_clusters}, coverage={coverage:.1f}%, "
    f"mcs={MIN_CLUSTER_SIZE}, ms={MIN_SAMPLES})",
    fontsize=12, fontweight="bold"
)

# ── 1: Barplot top-30 cluster terbesar ────────────────────────────────────────
ax = axes[0]
top30_ids   = [x[0] for x in sorted(cluster_sizes.items(), key=lambda x: x[1], reverse=True)[:30]]
top30_sizes = [cluster_sizes[cid] for cid in top30_ids]
colors = ["#2196F3" if sz >= MIN_CLUSTER_SIZE else "#FF9800" for sz in top30_sizes]
ax.bar(range(len(top30_sizes)), top30_sizes, color=colors, edgecolor="white", linewidth=0.5)
ax.axhline(MIN_CLUSTER_SIZE, color="red", linestyle="--", linewidth=1.5,
           label=f"min_cluster_size={MIN_CLUSTER_SIZE}")
ax.set_xlabel("Rank Cluster (terbesar → terkecil)", fontsize=10)
ax.set_ylabel("Jumlah Foto", fontsize=10)
ax.set_title("Top-30 Cluster Terbesar", fontsize=10, fontweight="bold")
ax.legend(fontsize=8)
ax.grid(axis="y", alpha=0.3)

# ── 2: Histogram ukuran cluster ───────────────────────────────────────────────
ax = axes[1]
ax.hist(sizes, bins=25, color="#2196F3", edgecolor="white", linewidth=0.5)
ax.axvline(sizes.mean(), color="orange", linestyle="--", linewidth=2,
           label=f"Mean={sizes.mean():.0f}")
ax.axvline(np.median(sizes), color="red", linestyle="-.", linewidth=2,
           label=f"Median={np.median(sizes):.0f}")
ax.set_xlabel("Ukuran Cluster (jumlah foto)", fontsize=10)
ax.set_ylabel("Frekuensi", fontsize=10)
ax.set_title("Distribusi Ukuran Cluster", fontsize=10, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# ── 3: Pie chart — coverage vs noise ─────────────────────────────────────────
ax = axes[2]
wedge_sizes = [n_clustered, n_noise]
wedge_labels = [
    f"Clustered\n{n_clustered:,} foto\n({coverage:.1f}%)",
    f"Noise\n{n_noise:,} foto\n({noise_pct:.1f}%)"
]
colors_pie = ["#2196F3", "#BDBDBD"]
explode    = (0.05, 0)
ax.pie(wedge_sizes, labels=wedge_labels, colors=colors_pie,
       explode=explode, autopct="%1.1f%%", startangle=90,
       textprops={"fontsize": 9})
ax.set_title("Coverage vs Noise", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "distribusi_cluster.png", dpi=150, bbox_inches="tight")
plt.show()
print("Tersimpan: distribusi_cluster.png")

## 6. Visualisasi UMAP 2D

UMAP digunakan **hanya untuk visualisasi** — bukan untuk clustering.  
Reduksi 512 → 2 dimensi dengan cosine metric.

In [ ]:
print("Menjalankan UMAP 2D (512 → 2 dim)...")
t0 = time.time()

reducer = umap.UMAP(
    n_components=2,
    metric="cosine",
    random_state=RANDOM_SEED,
    n_jobs=-1,
    verbose=False,
)
emb_2d = reducer.fit_transform(embeddings)

t_umap = time.time() - t0
print(f"UMAP selesai dalam {t_umap:.0f}s ({t_umap/60:.1f} menit)")
print(f"Shape output: {emb_2d.shape}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle(
    f"UMAP 2D — NB05 Final Clustering  "
    f"(n={n_clusters} clusters, coverage={coverage:.1f}%)",
    fontsize=13, fontweight="bold"
)

# ── 1: Scatter semua titik — warna per cluster ────────────────────────────────
ax = axes[0]
mask_noise = labels == -1
mask_clust = labels >= 0

# Noise — abu-abu di belakang
ax.scatter(
    emb_2d[mask_noise, 0], emb_2d[mask_noise, 1],
    c="#CCCCCC", s=2, alpha=0.4, label=f"Noise ({n_noise:,})", rasterized=True
)

# Cluster — warna berbeda
unique_labels = sorted(set(labels[mask_clust]))
cmap = plt.cm.get_cmap("tab20", min(20, n_clusters))
for i, lbl in enumerate(unique_labels):
    mask = labels == lbl
    ax.scatter(
        emb_2d[mask, 0], emb_2d[mask, 1],
        c=[cmap(i % 20)], s=3, alpha=0.7, rasterized=True
    )

ax.set_title(f"Semua {n_clusters} Cluster + Noise", fontsize=11, fontweight="bold")
ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
ax.legend(markerscale=4, fontsize=9)
ax.grid(alpha=0.2)

# ── 2: Scatter noise vs clustered (binary) ────────────────────────────────────
ax = axes[1]
ax.scatter(
    emb_2d[mask_noise, 0], emb_2d[mask_noise, 1],
    c="#FF5722", s=2, alpha=0.5, label=f"Noise ({noise_pct:.1f}%)", rasterized=True
)
ax.scatter(
    emb_2d[mask_clust, 0], emb_2d[mask_clust, 1],
    c="#2196F3", s=2, alpha=0.4, label=f"Clustered ({coverage:.1f}%)", rasterized=True
)
ax.set_title("Coverage vs Noise (UMAP)", fontsize=11, fontweight="bold")
ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
ax.legend(markerscale=4, fontsize=9)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "umap_2d.png", dpi=150, bbox_inches="tight")
plt.show()
print("Tersimpan: umap_2d.png")

## 7. Simpan Hasil

In [ ]:
# ── 1. Labels array ───────────────────────────────────────────────────────────
labels_path = OUTPUT_DIR / "labels.npy"
np.save(labels_path, labels)

# ── 2. Metadata dengan label cluster ─────────────────────────────────────────
metadata_labeled = []
for i, meta in enumerate(metadata):
    entry = dict(meta) if isinstance(meta, dict) else {"idx": i}
    entry["cluster_id"] = int(labels[i])
    entry["is_noise"]   = bool(labels[i] == -1)
    metadata_labeled.append(entry)

meta_path = OUTPUT_DIR / "metadata_labeled.pkl"
with open(meta_path, "wb") as f:
    pickle.dump(metadata_labeled, f)

# ── 3. Cluster summary ────────────────────────────────────────────────────────
cluster_summary = {
    "n_clusters"              : n_clusters,
    "n_noise"                 : n_noise,
    "n_clustered"             : n_clustered,
    "coverage_pct"            : round(coverage, 4),
    "noise_pct"               : round(noise_pct, 4),
    "cluster_sizes"           : dict(cluster_sizes),
    "size_mean"               : float(sizes.mean()),
    "size_median"             : float(np.median(sizes)),
    "size_min"                : int(sizes.min()),
    "size_max"                : int(sizes.max()),
    "silhouette_score"        : float(sil_score),
    "davies_bouldin_index"    : float(dbi_score),
    "subsample_size"          : int(n_sub),
    "random_seed"             : RANDOM_SEED,
    "params": {
        "min_cluster_size"        : MIN_CLUSTER_SIZE,
        "min_samples"             : MIN_SAMPLES,
        "cluster_selection_method": CLUSTER_SELECTION_METHOD,
        "metric"                  : "euclidean",
        "approx_min_span_tree"    : True,
    },
    "umap_2d": emb_2d,
}

summary_path = OUTPUT_DIR / "cluster_summary.pkl"
with open(summary_path, "wb") as f:
    pickle.dump(cluster_summary, f)

# ── 4. Ringkasan CSV ──────────────────────────────────────────────────────────
df_summary = pd.DataFrame([
    {"cluster_id": cid, "n_foto": sz}
    for cid, sz in sorted(cluster_sizes.items())
])
df_summary.to_csv(OUTPUT_DIR / "cluster_sizes.csv", index=False)

# ── Print ringkasan ────────────────────────────────────────────────────────────
print("=" * 55)
print("  OUTPUT NB05")
print("=" * 55)
for path in [labels_path, meta_path, summary_path,
             OUTPUT_DIR / "cluster_sizes.csv",
             OUTPUT_DIR / "distribusi_cluster.png",
             OUTPUT_DIR / "umap_2d.png"]:
    size_kb = path.stat().st_size / 1024 if path.exists() else 0
    print(f"  {path.name:<35} {size_kb:7.1f} KB")
print("=" * 55)
print()
print(f"  n_clusters   : {n_clusters}")
print(f"  coverage     : {coverage:.2f}%")
print(f"  Silhouette   : {sil_score:.4f}")
print(f"  DBI          : {dbi_score:.4f}")